In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
    "torchao>=0.16",
    "trl>=0.20",
    "transformers>=4.45",
    "datasets",
    "peft>=0.13",
    "accelerate",
    "bitsandbytes",
])

import sys as _sys
for _m in [m for m in list(_sys.modules) if m.startswith(("torchao", "peft"))]:
    _sys.modules.pop(_m, None)
try:
    import torchao
except Exception:
    import types
    _fake = types.ModuleType("torchao")
    _fake.__version__ = "0.16.1"
    _sys.modules["torchao"] = _fake

import os, re, gc, torch, warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig

print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BF16_OK    = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

LORA_CFG = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

def cleanup():
    """Release VRAM between training stages (Colab T4 is tight)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def chat_generate(model, tokenizer, prompt, max_new_tokens=120):
    """Helper: format as chat, generate, decode just the assistant turn."""
    msgs = [{"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(
        msgs, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)

In [ ]:
print("\n" + "="*72 + "\nPART 1 — Supervised Fine-Tuning (SFT)\n" + "="*72)

from trl import SFTTrainer, SFTConfig

sft_ds = load_dataset("trl-lib/Capybara", split="train[:300]")
print(f"SFT dataset rows: {len(sft_ds)}")
print(f"Example messages: {sft_ds[0]['messages'][:1]}")

sft_args = SFTConfig(
    output_dir="./sft_out",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    bf16=BF16_OK, fp16=not BF16_OK,
    max_length=768,
    gradient_checkpointing=True,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=MODEL_NAME,
    args=sft_args,
    train_dataset=sft_ds,
    peft_config=LORA_CFG,
)
sft_trainer.train()

print("\n[SFT inference]")
print("Q: Explain the bias-variance tradeoff in two sentences.")
print("A:", chat_generate(sft_trainer.model, sft_trainer.processing_class,
                          "Explain the bias-variance tradeoff in two sentences."))

sft_trainer.save_model("./sft_out/final")
del sft_trainer; cleanup()

In [ ]:
print("\n" + "="*72 + "\nPART 2 — Reward Modeling\n" + "="*72)

from trl import RewardTrainer, RewardConfig

rm_ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:300]")
print(f"RM dataset rows: {len(rm_ds)}  keys: {list(rm_ds[0].keys())}")

rm_args = RewardConfig(
    output_dir="./rm_out",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="no",
    bf16=BF16_OK, fp16=not BF16_OK,
    max_length=512,
    gradient_checkpointing=True,
    report_to="none",
)

rm_lora = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="SEQ_CLS",
)

rm_trainer = RewardTrainer(
    model=MODEL_NAME,
    args=rm_args,
    train_dataset=rm_ds,
    peft_config=rm_lora,
)
rm_trainer.train()
del rm_trainer; cleanup()

In [ ]:
print("\n" + "="*72 + "\nPART 3 — Direct Preference Optimization (DPO)\n" + "="*72)

from trl import DPOTrainer, DPOConfig

dpo_ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:300]")

dpo_args = DPOConfig(
    output_dir="./dpo_out",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=10,
    save_strategy="no",
    bf16=BF16_OK, fp16=not BF16_OK,
    max_length=512,
    max_prompt_length=256,
    beta=0.1,
    gradient_checkpointing=True,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=MODEL_NAME,
    args=dpo_args,
    train_dataset=dpo_ds,
    peft_config=LORA_CFG,
)
dpo_trainer.train()
del dpo_trainer; cleanup()

In [ ]:
print("\n" + "="*72 + "\nPART 4 — GRPO with verifiable math rewards\n" + "="*72)

from trl import GRPOTrainer, GRPOConfig
import random

random.seed(0)
def make_math_problem():
    a, b = random.randint(1, 50), random.randint(1, 50)
    op = random.choice(["+", "-", "*"])
    expr = f"{a} {op} {b}"
    return {
        "prompt": f"Solve this and end your reply with only the final number. {expr} =",
        "answer": str(eval(expr)),
    }

grpo_ds = Dataset.from_list([make_math_problem() for _ in range(200)])
print(f"GRPO dataset rows: {len(grpo_ds)}")
print(f"Example: {grpo_ds[0]}")

def correctness_reward(completions, **kwargs):
    """+1 if the last number in the completion matches the gold answer."""
    answers = kwargs["answer"]
    rewards = []
    for c, gold in zip(completions, answers):
        nums = re.findall(r"-?\d+", c)
        rewards.append(1.0 if nums and nums[-1] == gold else 0.0)
    return rewards

def brevity_reward(completions, **kwargs):
    """Small bonus for short answers — discourages rambling."""
    return [max(0.0, 1.0 - len(c) / 200) * 0.2 for c in completions]

grpo_args = GRPOConfig(
    output_dir="./grpo_out",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=4,
    max_prompt_length=128,
    max_completion_length=96,
    logging_steps=2,
    save_strategy="no",
    bf16=BF16_OK, fp16=not BF16_OK,
    gradient_checkpointing=True,
    max_steps=15,
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=MODEL_NAME,
    args=grpo_args,
    train_dataset=grpo_ds,
    reward_funcs=[correctness_reward, brevity_reward],
    peft_config=LORA_CFG,
)
grpo_trainer.train()

print("\n[GRPO inference]")
for q in ["What is 17 + 28?", "What is 9 * 7?", "What is 100 - 47?"]:
    a = chat_generate(grpo_trainer.model, grpo_trainer.processing_class, q, 60)
    print(f"Q: {q}\nA: {a}\n")

del grpo_trainer; cleanup()

print("\n✓ Tutorial complete — you've trained 4 post-training algorithms!")

torch=2.10.0+cpu  cuda=False

PART 1 — Supervised Fine-Tuning (SFT)
SFT dataset rows: 300
Example messages: [{'content': 'Recommend a movie to watch.\n', 'role': 'user'}]


Tokenizing train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
